# 03 - Sales Forecasting Portfolio

This notebook demonstrates **Predictive Analytics** for the Retail Multi-Brand project.  
We use historical daily net revenue to forecast future sales performance for the next 90 days.

**Goals:**
- Prepare daily time-series data from DuckDB marts. 
- Implement a time-series model (using **Facebook Prophet**).
- Visualize seasonal patterns (Weekly, Monthly, Yearly).
- Provide a forecast for business demand planning.

In [ ]:
# Install dependencies if needed
# !pip install prophet duckdb pandas matplotlib

import duckdb
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
import os

# Setup Paths
BASE_DIR = os.path.dirname(os.getcwd())
DB_PATH = os.path.join(BASE_DIR, 'data', 'multibrand_retail.duckdb')

def load_daily_sales():
    if not os.path.exists(DB_PATH):
        print("Error: Database not found. Please run ingestion/dbt first.")
        return None
    
    con = duckdb.connect(DB_PATH, read_only=True)
    # Aggregate sales by day
    query = """
        SELECT date::DATE as ds, SUM(net_amount) as y
        FROM fct_sales
        GROUP BY 1
        ORDER BY 1
    """
    df = con.execute(query).df()
    con.close()
    return df

## 1. Load & Inspect Time Series Data

In [ ]:
df = load_daily_sales()
if df is not None:
    print(f"Loaded {len(df)} days of sales data.")
    df.plot(x='ds', y='y', figsize=(12, 6), title='Actual Daily Net Revenue')
    plt.show()

## 2. Training the Prophet Model
Prophet is optimized for business time series which often have strong daily/weekly seasonality and holiday effects.

In [ ]:
if df is not None:
    # Initialize and fit model
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    m.add_country_holidays(country_name='US') # Optional: Adjust to your target region
    
    m.fit(df)
    
    # Create future timeframe (90 days)
    future = m.make_future_dataframe(periods=90)
    forecast = m.predict(future)
    
    print("Forecast complete.")

## 3. Visualize Forecast & Components

In [ ]:
if df is not None:
    # Plot the forecast
    fig1 = m.plot(forecast)
    plt.title("90-Day Sales Forecast")
    plt.xlabel("Date")
    plt.ylabel("Revenue")
    plt.show()
    
    # Plot components (Trend, Seasonality)
    fig2 = m.plot_components(forecast)
    plt.show()

## 4. Business Recommendations
Based on the forecast components:
- **Weekly Pattern**: If sales peak on weekends, consider launching promotions on Fridays.
- **Yearly Pattern**: Identify the 'Downturn' months to plan budget saving or aggressive marketing.
- **Growth Trend**: If the trend line is upward, the business is scaling healthily beyond seasonality.